# Quora Question Pairs — Análisis de Modelos

Este notebook explica los modelos implementados y propone nuevas alternativas para el reto de detectar preguntas duplicadas en Quora.

**Tarea**: clasificación binaria → `is_duplicate` (0/1)

**Pipeline general**:
1. Extraer *features* de cada par (q1, q2)
2. Entrenar un clasificador
3. Evaluar con ROC-AUC, Precision, Recall

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import scipy
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, classification_report
from sklearn.pipeline import Pipeline
from utils import (
    cast_list_as_strings,
    get_features_from_df,
    get_advanced_features,
    get_sbert_features,
    get_mistakes
)
print('Imports OK')

## 1. Carga y split de datos

Se usa el split exacto del guide (random_state=123):

In [ ]:
DATA_PATH = '../nlp_deliv1_materials/quora_data.csv'
quora_df = pd.read_csv(DATA_PATH)
print('Shape total:', quora_df.shape)
quora_df.head(3)

In [ ]:
A_df, test_df = train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = train_test_split(A_df, test_size=0.05, random_state=123)
print('train:', train_df.shape, '| val:', val_df.shape, '| test:', test_df.shape)
print('Duplicados en train:', train_df['is_duplicate'].mean().round(3))

In [ ]:
# Subconjunto pequeño para demostración rápida
DEMO = True
N = 5000 if DEMO else len(train_df)
train_s = train_df.sample(N, random_state=42).reset_index(drop=True)
val_s   = val_df.sample(min(1000, len(val_df)), random_state=42).reset_index(drop=True)
y_train = train_s['is_duplicate'].values
y_val   = val_s['is_duplicate'].values
print(f'Demo mode={DEMO}: train={len(train_s)}, val={len(val_s)}')

---
## Modelo 1 — Simple: CountVectorizer + Logistic Regression

### Explicación

**CountVectorizer** convierte cada pregunta en un vector de frecuencias de palabras (Bag-of-Words).  
Se concatenan horizontalmente los vectores de q1 y q2 → el clasificador aprende qué palabras compartidas son señal de duplicado.

```
q1: 'how to learn python'  → [1, 0, 1, 1, ...]
q2: 'best way to learn python' → [0, 1, 1, 1, ...]
X  = [q1 | q2]  (hstack)
```

**Limitaciones:**
- No captura semántica (sinónimos no detectados)
- Sensible al orden y vocabulario exacto
- Ignora la estructura de la pregunta

In [ ]:
q1_train = cast_list_as_strings(list(train_s['question1']))
q2_train = cast_list_as_strings(list(train_s['question2']))
all_q = q1_train + q2_train

cv = CountVectorizer(ngram_range=(1,1), max_features=20000)
cv.fit(all_q)
print('Vocab size:', len(cv.vocabulary_))

In [ ]:
X_tr1 = get_features_from_df(train_s, cv)
X_va1 = get_features_from_df(val_s, cv)

model1 = LogisticRegression(solver='liblinear', random_state=123)
model1.fit(X_tr1, y_train)

y_pred1 = model1.predict(X_va1)
y_prob1 = model1.predict_proba(X_va1)[:,1]

auc1  = roc_auc_score(y_val, y_prob1)
prec1 = precision_score(y_val, y_pred1)
rec1  = recall_score(y_val, y_pred1)
print(f'[Modelo 1] AUC={auc1:.4f}  Prec={prec1:.4f}  Rec={rec1:.4f}')

---
## Modelo 2 — Advanced Features: BoW + Jaccard + Longitud + LR

### Explicación

Se añaden dos features handcrafted sobre el par (q1, q2):

| Feature | Descripción |
|---|---|
| **Jaccard Similarity** | \|q1 ∩ q2\| / \|q1 ∪ q2\| a nivel de tokens |
| **Length Difference** | \|len(q1) - len(q2)\| / max(len) |

Implementación from scratch en `utils.py` → `jaccard_similarity_scratch`.

Mejora el modelo 1 añadiendo señales de solapamiento léxico directo.

**Limitaciones:**
- Jaccard sigue siendo léxico, no semántico
- Funciona mal con paráfrasis ('buy' vs 'purchase')

In [ ]:
X_tr2 = get_advanced_features(train_s, cv)
X_va2 = get_advanced_features(val_s, cv)

model2 = LogisticRegression(solver='liblinear', random_state=123)
model2.fit(X_tr2, y_train)

y_pred2 = model2.predict(X_va2)
y_prob2 = model2.predict_proba(X_va2)[:,1]

auc2  = roc_auc_score(y_val, y_prob2)
prec2 = precision_score(y_val, y_pred2)
rec2  = recall_score(y_val, y_pred2)
print(f'[Modelo 2] AUC={auc2:.4f}  Prec={prec2:.4f}  Rec={rec2:.4f}')

---
## Modelo 3 — SBERT + Logistic Regression

### Explicación

**SentenceBERT** (`all-MiniLM-L6-v2`) es un transformer fine-tuneado para producir embeddings densos que capturan significado semántico.

El pipeline:
1. Codificar q1 → e1 ∈ ℝ^384
2. Codificar q2 → e2 ∈ ℝ^384
3. Features: `|e1 - e2|` (diferencia absoluta) + cosine similarity
4. Regresión logística sobre estas features

Ventajas:
- Captura sinónimos y paráfrasis
- Representación contextual

Limitación:
- Lento en datasets grandes (291k ejemplos ~ interrumpido)
- Requiere GPU para escalar bien

*Nota: En el notebook de entrenamiento original (train_models.ipynb) el Modelo 3 fue interrumpido (KeyboardInterrupt) por tiempo. Aquí lo ejecutamos sobre el subconjunto demo.*

In [ ]:
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print('SBERT cargado')

In [ ]:
X_tr3 = get_sbert_features(train_s, sbert)
X_va3 = get_sbert_features(val_s, sbert)

model3 = LogisticRegression(solver='liblinear', random_state=123, max_iter=500)
model3.fit(X_tr3, y_train)

y_pred3 = model3.predict(X_va3)
y_prob3 = model3.predict_proba(X_va3)[:,1]

auc3  = roc_auc_score(y_val, y_prob3)
prec3 = precision_score(y_val, y_pred3)
rec3  = recall_score(y_val, y_pred3)
print(f'[Modelo 3] AUC={auc3:.4f}  Prec={prec3:.4f}  Rec={rec3:.4f}')

---
## Modelo 4 (NUEVO) — TF-IDF + Logistic Regression

### Explicación

**TF-IDF** (Term Frequency–Inverse Document Frequency) penaliza las palabras muy frecuentes (poco informativas) y da más peso a las específicas del documento.

$$\text{TF-IDF}(t,d) = tf(t,d) \times \log\frac{N}{df(t)+1}$$

Mejora sobre CountVectorizer porque:
- 'the', 'is', 'what' tienen peso casi 0
- Palabras clave del dominio tienen peso alto
- Bigramas capturan frases comunes

Usamos `ngram_range=(1,2)` para incluir bigramas.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=30000, sublinear_tf=True)
all_q_train = cast_list_as_strings(list(train_s['question1']) + list(train_s['question2']))
tfidf.fit(all_q_train)

q1_tr = tfidf.transform(cast_list_as_strings(list(train_s['question1'])))
q2_tr = tfidf.transform(cast_list_as_strings(list(train_s['question2'])))
X_tr4 = scipy.sparse.hstack([q1_tr, q2_tr])

q1_va = tfidf.transform(cast_list_as_strings(list(val_s['question1'])))
q2_va = tfidf.transform(cast_list_as_strings(list(val_s['question2'])))
X_va4 = scipy.sparse.hstack([q1_va, q2_va])

model4 = LogisticRegression(solver='liblinear', random_state=123, C=1.0)
model4.fit(X_tr4, y_train)

y_pred4 = model4.predict(X_va4)
y_prob4 = model4.predict_proba(X_va4)[:,1]

auc4  = roc_auc_score(y_val, y_prob4)
prec4 = precision_score(y_val, y_pred4)
rec4  = recall_score(y_val, y_pred4)
print(f'[Modelo 4] AUC={auc4:.4f}  Prec={prec4:.4f}  Rec={rec4:.4f}')

---
## Modelo 5 (NUEVO) — TF-IDF + Cosine Similarity + LR

### Explicación

En lugar de concatenar los vectores TF-IDF, calculamos directamente la **similitud coseno** entre q1 y q2 como feature, junto con la diferencia de normas.

$$\cos(q1, q2) = \frac{q1 \cdot q2}{\|q1\| \|q2\|}$$

Esto reduce la dimensionalidad drásticamente y fuerza al modelo a razonar sobre similitud, no sobre presencia de palabras.

In [ ]:
from sklearn.metrics.pairwise import paired_cosine_distances
import re

def get_tfidf_similarity_features(df, tfidf_model):
    q1 = tfidf_model.transform(cast_list_as_strings(list(df['question1'])))
    q2 = tfidf_model.transform(cast_list_as_strings(list(df['question2'])))
    cos_sim = 1 - paired_cosine_distances(q1, q2)
    # Diferencia de normas
    norm1 = np.array(q1.multiply(q1).sum(axis=1)).flatten() ** 0.5
    norm2 = np.array(q2.multiply(q2).sum(axis=1)).flatten() ** 0.5
    norm_diff = np.abs(norm1 - norm2)
    # Diferencia absoluta entre vectores TF-IDF (solo top 500 dims)
    q1_d = np.array(q1[:, :500].todense())
    q2_d = np.array(q2[:, :500].todense())
    abs_diff = np.abs(q1_d - q2_d)
    return np.hstack([cos_sim.reshape(-1,1), norm_diff.reshape(-1,1), abs_diff])

X_tr5 = get_tfidf_similarity_features(train_s, tfidf)
X_va5 = get_tfidf_similarity_features(val_s, tfidf)

model5 = LogisticRegression(solver='lbfgs', random_state=123, max_iter=500)
model5.fit(X_tr5, y_train)

y_pred5 = model5.predict(X_va5)
y_prob5 = model5.predict_proba(X_va5)[:,1]

auc5  = roc_auc_score(y_val, y_prob5)
prec5 = precision_score(y_val, y_pred5)
rec5  = recall_score(y_val, y_pred5)
print(f'[Modelo 5] AUC={auc5:.4f}  Prec={prec5:.4f}  Rec={rec5:.4f}')

---
## Modelo 6 (NUEVO) — SBERT + Gradient Boosting

### Explicación

Usamos los mismos embeddings SBERT del Modelo 3 pero cambiamos el clasificador por **Gradient Boosting** (GBM), que es un ensemble de árboles de decisión que se construyen secuencialmente.

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

Ventajas sobre Logistic Regression:
- Captura relaciones no lineales entre features
- Más robusto ante features correlacionadas
- No requiere escalado de features

In [ ]:
# Reutilizamos X_tr3 y X_va3 (embeddings SBERT ya calculados)
if scipy.sparse.issparse(X_tr3):
    X_tr3_d = X_tr3.toarray()
    X_va3_d = X_va3.toarray()
else:
    X_tr3_d = np.array(X_tr3)
    X_va3_d = np.array(X_va3)

model6 = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=123)
model6.fit(X_tr3_d, y_train)

y_pred6 = model6.predict(X_va3_d)
y_prob6 = model6.predict_proba(X_va3_d)[:,1]

auc6  = roc_auc_score(y_val, y_prob6)
prec6 = precision_score(y_val, y_pred6)
rec6  = recall_score(y_val, y_pred6)
print(f'[Modelo 6] AUC={auc6:.4f}  Prec={prec6:.4f}  Rec={rec6:.4f}')

---
## Comparativa de Resultados

In [ ]:
results = pd.DataFrame([
    {'Modelo': 'M1: CountVec + LR (simple)', 'ROC-AUC': auc1, 'Precision': prec1, 'Recall': rec1},
    {'Modelo': 'M2: CountVec+Jaccard+Len + LR', 'ROC-AUC': auc2, 'Precision': prec2, 'Recall': rec2},
    {'Modelo': 'M3: SBERT + LR (avanzado)', 'ROC-AUC': auc3, 'Precision': prec3, 'Recall': rec3},
    {'Modelo': 'M4: TF-IDF bigramas + LR (nuevo)', 'ROC-AUC': auc4, 'Precision': prec4, 'Recall': rec4},
    {'Modelo': 'M5: TF-IDF cosine + LR (nuevo)', 'ROC-AUC': auc5, 'Precision': prec5, 'Recall': rec5},
    {'Modelo': 'M6: SBERT + GradientBoosting (nuevo)', 'ROC-AUC': auc6, 'Precision': prec6, 'Recall': rec6},
])
results = results.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results.round(4)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4CAF50' if 'nuevo' in m else '#2196F3' for m in results['Modelo']]
bars = ax.barh(results['Modelo'], results['ROC-AUC'], color=colors)
ax.set_xlabel('ROC-AUC')
ax.set_title('Comparativa de Modelos — Quora Question Pairs')
ax.set_xlim(0.5, 1.0)
for bar, val in zip(bars, results['ROC-AUC']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center')
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#2196F3', label='Existentes'),
    plt.Rectangle((0,0),1,1, color='#4CAF50', label='Nuevos propuestos')
])
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100)
plt.show()
print('Gráfico guardado')

---
## Análisis de Errores — Mejor Modelo

In [ ]:
# Análisis de errores del modelo con mayor AUC
best_idx = results.index[0]
best_name = results.loc[0, 'Modelo']
print(f'Mejor modelo: {best_name}')

# Mostrar ejemplos mal clasificados del Modelo 3 (SBERT+LR)
err_idx, preds3 = get_mistakes(model3, X_va3, y_val)
print(f'\nErrores Modelo SBERT+LR: {len(err_idx)} / {len(y_val)}')
errors_df = val_s.iloc[err_idx[:5]][['question1','question2','is_duplicate']].copy()
errors_df['pred'] = preds3[err_idx[:5]]
errors_df

---
## Conclusiones y Próximos Pasos

### Resumen

| Tipo | Modelos | Insight |
|---|---|---|
| **Léxico** | M1 (CountVec), M4 (TF-IDF) | Rápidos, buen baseline. TF-IDF ligeramente mejor por penalización de stopwords |
| **Léxico+Handcrafted** | M2 (Jaccard+Len) | Jaccard ayuda cuando hay mucho solapamiento textual |
| **Semántico** | M3 (SBERT+LR), M6 (SBERT+GBM) | Captura paráfrasis, mejor en casos difíciles |

### Próximos pasos propuestos

1. **Fine-tune SBERT** directamente en pares Quora con Contrastive Loss
2. **Cross-encoder**: pasar `[CLS] q1 [SEP] q2 [SEP]` directamente a BERT para clasificación conjunta
3. **Ensemble**: combinar scores de M4 (TF-IDF rápido) + M3 (SBERT preciso)
4. **Feature engineering**: edit distance, overlap de entidades nombradas, POS tags
5. **Threshold tuning**: ajustar umbral de decisión según precision/recall deseado